# Passes

Detects pass events from `ball_frame_table`'s carrier transitions and
reports **all passes** and **completed passes** for both teams. Reads only
`player_frame_table` / `ball_frame_table` (built by `match_frame_table.ipynb`)
-- never the raw per-stage caches.

**Definition used here** (this is a proxy, not ground truth -- worth being
explicit about since there's no whistle/event feed to check against):
a *pass* is a transition of ball possession from one player's continuous
carrier segment to a different player's, close enough in time to be one
continuous phase of play. A pass is **completed** if the receiver is on the
same team as the passer, and a **turnover** if not. Consecutive frames where
the *same* player regains the carrier tag after a brief tracking blip are
merged into one segment, not counted as a pass.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

import paths

for p in (paths.PLAYER_FRAME_TABLE_CACHE_PATH, paths.BALL_FRAME_TABLE_CACHE_PATH):
    if not Path(p).exists():
        raise FileNotFoundError(
            f"{p} not found -- run match_frame_table.ipynb first, it builds both frame tables."
        )

player_frame_table = pd.read_parquet(paths.PLAYER_FRAME_TABLE_CACHE_PATH)
ball_frame_table = pd.read_parquet(paths.BALL_FRAME_TABLE_CACHE_PATH)

print("player_frame_table:", player_frame_table.shape)
print("ball_frame_table:  ", ball_frame_table.shape)

player_frame_table: (70396, 10)
ball_frame_table:   (3001, 10)


## Team lookup

A track's team is effectively constant across the match, so this collapses
`player_frame_table` down to one `track_id -> team` mapping (majority vote,
in case of a rare mis-assigned frame). Referees have no team and are
excluded -- they never appear as a carrier anyway, since `ball_tracker`'s
carrier assigner only considers player/goalkeeper tracks.

In [2]:
team_by_id = (
    player_frame_table.dropna(subset=["team"])
    .groupby("track_id")["team"]
    .agg(lambda s: s.mode().iat[0])
    .astype(int)
    .to_dict()
)
print(f"{len(team_by_id)} tracks with a team assigned")

22 tracks with a team assigned


## Possession segments

Collapse `ball_frame_table.carrier_track_id` into runs: `(track_id,
start_frame, end_frame, n_frames)`. A new segment starts whenever the
carrier changes **or** whenever there's a gap in frame coverage (carrier
was unassigned in between) -- so two same-player runs separated by a gap
stay two segments unless later merged by the pass-linking step below, and
a same-player run interrupted by a single missing frame (no gap) is *not*
artificially split.

In [3]:
def build_possession_segments(ball_frame_table):
    df = ball_frame_table[["frame_idx", "carrier_track_id"]].dropna(subset=["carrier_track_id"]).copy()
    df["carrier_track_id"] = df["carrier_track_id"].astype(int)
    if df.empty:
        return pd.DataFrame(columns=["track_id", "start_frame", "end_frame", "n_frames"])

    changed = df["carrier_track_id"] != df["carrier_track_id"].shift()
    gapped = df["frame_idx"] != (df["frame_idx"].shift() + 1)
    segment_id = (changed | gapped).cumsum()

    segments = (
        df.groupby(segment_id)
        .agg(track_id=("carrier_track_id", "first"),
             start_frame=("frame_idx", "min"),
             end_frame=("frame_idx", "max"),
             n_frames=("frame_idx", "size"))
        .reset_index(drop=True)
    )
    return segments


MIN_SEGMENT_FRAMES = 3  # drop segments shorter than this -- matches CarrierConfig.min_frames_to_switch,
                         # anything shorter than the carrier assigner's own switch hysteresis is noise

segments = build_possession_segments(ball_frame_table)
n_before = len(segments)
segments = segments[segments["n_frames"] >= MIN_SEGMENT_FRAMES].reset_index(drop=True)
print(f"Possession segments: {n_before} -> {len(segments)} after dropping runs shorter than {MIN_SEGMENT_FRAMES} frames")
segments.head()

Possession segments: 65 -> 65 after dropping runs shorter than 3 frames


,track_id,start_frame,end_frame,n_frames
0,13,2,64,63
1,18,72,109,38
2,9,124,155,32
3,20,156,167,12
4,23,179,181,3


## Pass events

Link adjacent segments into pass events. Two segments are linked as a pass
when: the carrier actually changed (not the same player reappearing after
a blip), and the gap between the passer's last frame and the receiver's
first frame is short enough to plausibly be one continuous pass rather than
a stoppage or restart.

`MAX_GAP_FRAMES` is deliberately generous (3s @ 25fps) -- long balls and
crosses routinely lose ball-tracking mid-flight (small, fast, sometimes
airborne), and a long pass shouldn't be thrown out just because the
Kalman/RTS tracker had a gap during it. A multi-second gap, on the other
hand, is far more likely a throw-in, goal kick, or other restart than a
single pass.

In [4]:
MAX_GAP_FRAMES = 75  # ~3s @ ball_tracker.FPS -- tune against your actual footage's fps if it isn't 25


def build_pass_events(segments, team_by_id, max_gap_frames=MAX_GAP_FRAMES):
    events = []
    for i in range(len(segments) - 1):
        passer = segments.iloc[i]
        receiver = segments.iloc[i + 1]

        if passer["track_id"] == receiver["track_id"]:
            continue  # same player -- a tracking blip mid-possession, not a pass

        gap = receiver["start_frame"] - passer["end_frame"]
        if gap > max_gap_frames:
            continue  # too long to attribute to one continuous pass

        passer_team = team_by_id.get(passer["track_id"])
        receiver_team = team_by_id.get(receiver["track_id"])
        if passer_team is None or receiver_team is None:
            continue  # can't classify outcome without both teams known

        events.append({
            "pass_frame": int(passer["end_frame"]),
            "receive_frame": int(receiver["start_frame"]),
            "gap_frames": int(gap),
            "passer_id": int(passer["track_id"]),
            "receiver_id": int(receiver["track_id"]),
            "passer_team": passer_team,
            "receiver_team": receiver_team,
            "completed": passer_team == receiver_team,
        })

    return pd.DataFrame(events, columns=[
        "pass_frame", "receive_frame", "gap_frames",
        "passer_id", "receiver_id", "passer_team", "receiver_team", "completed",
    ])


pass_events = build_pass_events(segments, team_by_id)
print(f"{len(pass_events)} pass events detected")
pass_events.head(10)

62 pass events detected


,pass_frame,receive_frame,gap_frames,passer_id,receiver_id,passer_team,receiver_team,completed
0,64,72,8,13,18,1,1,True
1,109,124,15,18,9,1,1,True
2,155,156,1,9,20,1,0,False
3,167,179,12,20,23,0,1,False
4,181,182,1,23,17,1,0,False
5,192,204,12,17,21,0,1,False
6,227,228,1,21,22,1,0,False
7,235,236,1,22,17,0,0,True
8,244,245,1,17,23,0,1,False
9,278,279,1,23,15,1,0,False


## Sanity checks

In [5]:
print(f"Total pass events      -> {len(pass_events)}")
print(f"Completed               -> {pass_events['completed'].sum()} ({pass_events['completed'].mean()*100:.1f}%)")
print(f"Turnovers                -> {(~pass_events['completed']).sum()}")
print()
print("gap_frames distribution (time between passer losing it and receiver gaining it):")
print(pass_events["gap_frames"].describe())
print()
print("Passes per passer team:")
print(pass_events["passer_team"].value_counts().sort_index())

Total pass events      -> 62
Completed               -> 32 (51.6%)
Turnovers                -> 30

gap_frames distribution (time between passer losing it and receiver gaining it):
count    62.000000
mean      8.451613
std       9.415156
min       1.000000
25%       1.000000
50%       5.500000
75%      13.750000
max      38.000000
Name: gap_frames, dtype: float64

Passes per passer team:
passer_team
0    35
1    27
Name: count, dtype: int64


## All passes and completed passes by team

`attempted` counts every pass event where that team had the ball at the
start of the transition, regardless of outcome. `completed` is the
same-team-receiver subset.

In [6]:
team_summary = (
    pass_events.groupby("passer_team")
    .agg(attempted=("completed", "size"), completed=("completed", "sum"))
    .reset_index()
    .rename(columns={"passer_team": "team"})
)
team_summary["completion_pct"] = (team_summary["completed"] / team_summary["attempted"] * 100).round(1)

print("Per team:")
print(team_summary.to_string(index=False))
print()
print(f"All passes (both teams)       -> {team_summary['attempted'].sum()}")
print(f"Completed passes (both teams) -> {team_summary['completed'].sum()}")
team_summary

Per team:
 team  attempted  completed  completion_pct
    0         35         20            57.1
    1         27         12            44.4

All passes (both teams)       -> 62
Completed passes (both teams) -> 32


,team,attempted,completed,completion_pct
0,0,35,20,57.1
1,1,27,12,44.4


## Top passers by team

Same `pass_events` table, grouped one level deeper: `(passer_team,
passer_id)` instead of just `passer_team`. `TOP_N_PASSERS` caps how many
players are shown per team, ranked by attempted passes.

In [7]:
TOP_N_PASSERS = 5

passer_stats = (
    pass_events.groupby(["passer_team", "passer_id"])
    .agg(attempted=("completed", "size"), completed=("completed", "sum"))
    .reset_index()
)
passer_stats["completion_pct"] = (passer_stats["completed"] / passer_stats["attempted"] * 100).round(1)

top_passers = (
    passer_stats.sort_values(["passer_team", "attempted"], ascending=[True, False])
    .groupby("passer_team")
    .head(TOP_N_PASSERS)
    .reset_index(drop=True)
)

for team, group in top_passers.groupby("passer_team"):
    print(f"\nTop passers -- team {team}:")
    print(group[["passer_id", "attempted", "completed", "completion_pct"]].to_string(index=False))

top_passers


Top passers -- team 0:
 passer_id  attempted  completed  completion_pct
        10          7          6            85.7
        22          5          2            40.0
        17          4          2            50.0
        12          3          2            66.7
        15          3          1            33.3

Top passers -- team 1:
 passer_id  attempted  completed  completion_pct
        23          5          3            60.0
        13          4          1            25.0
        18          4          4           100.0
         9          3          0             0.0
        11          3          2            66.7


,passer_team,passer_id,attempted,completed,completion_pct
0,0,10,7,6,85.7
1,0,22,5,2,40.0
2,0,17,4,2,50.0
3,0,12,3,2,66.7
4,0,15,3,1,33.3
5,1,23,5,3,60.0
6,1,13,4,1,25.0
7,1,18,4,4,100.0
8,1,9,3,0,0.0
9,1,11,3,2,66.7


## Cache as parquet

Same load-or-build pattern as every other stage.

In [8]:
FORCE_REBUILD_PASSES = False


def get_or_build_pass_events(force_rebuild=FORCE_REBUILD_PASSES):
    cache_path = Path(paths.PASS_EVENTS_CACHE_PATH)
    if cache_path.exists() and not force_rebuild:
        print(f"\u2705 Loaded pass events from cache.")
        return pd.read_parquet(cache_path)

    segments = build_possession_segments(ball_frame_table)
    segments = segments[segments["n_frames"] >= MIN_SEGMENT_FRAMES].reset_index(drop=True)
    events = build_pass_events(segments, team_by_id)

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    events.to_parquet(cache_path, index=False)
    print(f"\U0001F4BE Saved pass events to cache.")
    return events


pass_events = get_or_build_pass_events()

💾 Saved pass events to cache.


## Next steps

`pressure.py` and `possession.py` can both read `pass_events` alongside
`player_frame_table` / `ball_frame_table` -- e.g. pressure that immediately
follows a turnover (a counter-press) is a distinct, interesting stat once
both tables exist.